# Analisis Data Menggunakan Naive Bayes (KNIME + Python)



## 1. Pendahuluan

Metode Naive Bayes merupakan salah satu algoritma klasifikasi yang banyak digunakan dalam data mining dan machine learning. Algoritma ini bekerja berdasarkan konsep probabilitas dan teorema Bayes dengan asumsi bahwa setiap fitur bersifat independen satu sama lain. Meskipun asumsi tersebut cukup sederhana, Naive Bayes sering menghasilkan performa yang baik dalam berbagai kasus klasifikasi.



## 2. Konsep Naive Bayes

Metode yang digunakan pada analisis ini adalah Naive Bayes. Secara sederhana, metode ini bekerja dengan menghitung peluang suatu data masuk ke dalam kelas tertentu berdasarkan fitur yang dimilikinya. Setelah semua peluang dihitung, model akan memilih kelas dengan nilai probabilitas terbesar sebagai hasil prediksi.

Dasar dari metode ini adalah Teorema Bayes, yang dirumuskan sebagai berikut:

$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$

Namun dalam praktiknya, karena nilai $(P(X))$ sama untuk semua kelas, perbandingan cukup dilakukan pada:

$P(C|X) = P(C) \cdot P(X|C)$


Jika fitur lebih dari satu, maka dihitung:

$P(C|X) = P(C) \cdot P(x_1|C) \cdot P(x_2|C) \cdot ... \cdot P(x_n|C)$



## 3. Asumsi Naive Bayes

Naive Bayes memiliki asumsi bahwa setiap fitur tidak saling bergantung. Artinya, setiap variabel dianggap memiliki pengaruh sendiri terhadap hasil, tanpa dipengaruhi oleh variabel lain.

Sebagai contoh, model menganggap bahwa umur tidak berhubungan dengan total pengeluaran, atau lama berlangganan tidak berhubungan dengan frekuensi penggunaan. Walaupun dalam kenyataannya bisa saja ada hubungan, asumsi ini tetap digunakan agar proses perhitungan menjadi lebih sederhana.


## 4. Gaussian Naive Bayes

Karena sebagian besar data berbentuk numerik, maka digunakan Gaussian Naive Bayes. Metode ini mengasumsikan bahwa data mengikuti distribusi normal.

Rumus distribusi Gaussian:

$P(x_i|C) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x_i-\mu)^2}{2\sigma^2}}$

Dengan pendekatan ini, model menghitung seberapa dekat nilai suatu fitur terhadap rata-rata kelas tertentu.


## 5. Preprocessing Data

Sebelum data digunakan dalam model, dilakukan beberapa tahap preprocessing.

### 5.1 Missing Value

Data yang kosong perlu ditangani karena tidak bisa langsung diproses oleh model. Pada tahap ini:

- Data numerik diisi menggunakan nilai rata-rata (mean)
- Data kategori diisi dengan nilai yang paling sering muncul
- Kolom target (Churn) tidak diubah sembarangan

Rumus mean:

$
Mean = \frac{\sum x}{n}
$

Dengan cara ini, data tetap bisa digunakan tanpa harus banyak menghapus baris.

![](https://cdn.mathpix.com/snip/images/91p-rm5yQDZfQKuyZrQ5KFH6koir3BI0NUBaHbeyIpQ.original.fullsize.png)

### 5.2 Normalisasi

Normalisasi dilakukan untuk menyamakan skala antar fitur. Hal ini penting karena setiap fitur memiliki rentang nilai yang berbeda.

Rumus Min-Max Normalization:

$
X' = \frac{X - X_{min}}{X_{max} - X_{min}}
$

Dengan normalisasi, semua fitur berada pada skala yang lebih seimbang sehingga model dapat bekerja lebih optimal.

![](https://cdn.mathpix.com/snip/images/j-jmFih9hMUdm0opgSsyRHyQz8YjoTDBl-oG8-evUB8.original.fullsize.png)


## 6. Workflow KNIME

Alur proses yang digunakan adalah:

```text
CSV Reader => Missing Value => Color Manager => Number to String => Normalizer => Python Script => Scorer
```
![](https://cdn.mathpix.com/snip/images/bXQhfYsEvma0Xa0Xp9GSbxso82AES3wRnSlhp05gZeA.original.fullsize.png)



## 7. Penjelasan Node

### a. CSV Reader  
Digunakan untuk membaca dataset dari file CSV. Pada tahap ini data dimasukkan ke dalam KNIME.

![](https://cdn.mathpix.com/snip/images/9Ep0k8XP2H1wixbTHKItHfhpBFAQ9fPNsh-O66UaGcU.original.fullsize.png)

### b. Missing Value  
Digunakan untuk menangani data kosong. Nilai kosong diisi agar tidak mengganggu proses modeling.

![](https://cdn.mathpix.com/snip/images/91p-rm5yQDZfQKuyZrQ5KFH6koir3BI0NUBaHbeyIpQ.original.fullsize.png)

### c. Color Manager  
Digunakan untuk memberi warna pada data berdasarkan kolom Churn. Fungsinya hanya untuk visualisasi.

![](https://cdn.mathpix.com/snip/images/mQUefdvf2mYLagaTFwar7iDTb7GsqHA1h3xJfm2-zMw.original.fullsize.png)

### d. Number to String  
Mengubah kolom Churn menjadi tipe string agar dikenali sebagai kategori.

![](https://cdn.mathpix.com/snip/images/P6NZQR6DY5yfmBc6LysR0FQXIjUDasp2mtKsw-j5jR4.original.fullsize.png)

### e. Normalizer  
Digunakan untuk menyamakan skala data numerik agar tidak ada fitur yang terlalu dominan.


![](https://cdn.mathpix.com/snip/images/j-jmFih9hMUdm0opgSsyRHyQz8YjoTDBl-oG8-evUB8.original.fullsize.png)

### f. Python Script

Node **Python Script** digunakan untuk membuat model **Gaussian Naive Bayes** menggunakan library `scikit-learn`. Pada node ini, data dari KNIME diambil lalu diproses menggunakan Python. Proses yang dilakukan meliputi pemilihan fitur, pemisahan target, pembagian data training dan testing, pembuatan model, prediksi, serta perhitungan probabilitas.

Kode yang digunakan pada node Python Script adalah sebagai berikut:

```python
import knime.scripting.io as knio
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

df = knio.input_tables[0].to_pandas()

features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]

df = df.dropna(subset=features + ["Churn"])

X = df[features]
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

model = GaussianNB()
model.fit(X_train, y_train)

prediction = model.predict(X_test)
probability = model.predict_proba(X_test)

output = pd.DataFrame({
    "Actual": y_test.values,
    "Prediction": prediction,
    "Prob_0": probability[:, 0],
    "Prob_1": probability[:, 1]
})

knio.output_tables[0] = knio.Table.from_pandas(output)
```

#### Penjelasan Kode

Bagian pertama digunakan untuk memanggil library yang dibutuhkan.

```python
import knime.scripting.io as knio
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
```

`knime.scripting.io` digunakan agar Python dapat mengambil data dari KNIME dan mengirim hasilnya kembali ke KNIME. Library `pandas` digunakan untuk mengolah data dalam bentuk tabel. `train_test_split` digunakan untuk membagi data menjadi data latih dan data uji. `GaussianNB` digunakan untuk membuat model Gaussian Naive Bayes.

```python
df = knio.input_tables[0].to_pandas()
```

Kode ini mengambil data dari input Python Script di KNIME. Data tersebut kemudian diubah menjadi DataFrame pandas agar lebih mudah diproses di Python.

```python
features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]
```

Bagian ini menentukan fitur yang digunakan untuk memprediksi churn. Fitur yang dipakai adalah data numerik yang dianggap berpengaruh terhadap kemungkinan pelanggan churn.

```python
df = df.dropna(subset=features + ["Churn"])
```

Kode ini digunakan untuk menghapus baris yang masih memiliki nilai kosong pada fitur atau target. Walaupun missing value sudah ditangani di KNIME, baris ini tetap digunakan sebagai pengaman agar model tidak error.

```python
X = df[features]
y = df["Churn"]
```

`X` berisi fitur yang digunakan sebagai input model, sedangkan `y` berisi target yang akan diprediksi, yaitu kolom `Churn`.

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)
```

Kode ini membagi data menjadi data training dan testing. Data training digunakan untuk melatih model, sedangkan data testing digunakan untuk menguji hasil prediksi. Nilai `test_size=0.3` berarti 30% data digunakan sebagai data testing dan 70% sebagai data training. `random_state=42` digunakan agar hasil pembagian data tetap sama setiap kali dijalankan. `stratify=y` digunakan agar proporsi kelas `0` dan `1` tetap seimbang pada data training dan testing.

```python
model = GaussianNB()
model.fit(X_train, y_train)
```

Bagian ini membuat model Gaussian Naive Bayes dan melatihnya menggunakan data training. Pada proses ini model mempelajari pola dari fitur pelanggan terhadap target `Churn`.

```python
prediction = model.predict(X_test)
```

Kode ini digunakan untuk memprediksi data testing. Hasilnya berupa kelas akhir, yaitu `0` atau `1`.

```python
probability = model.predict_proba(X_test)
```

Kode ini digunakan untuk menghasilkan probabilitas setiap kelas. Output probabilitas ini menunjukkan seberapa besar peluang data masuk ke kelas `0` atau kelas `1`.

Contoh:

| Prob_0 | Prob_1 | Prediction |
|---:|---:|---|
| 0.138 | 0.862 | 1 |
| 0.905 | 0.095 | 0 |

Jika `Prob_1` lebih besar daripada `Prob_0`, maka model memprediksi kelas `1`. Sebaliknya, jika `Prob_0` lebih besar, maka model memprediksi kelas `0`.

```python
output = pd.DataFrame({
    "Actual": y_test.values,
    "Prediction": prediction,
    "Prob_0": probability[:, 0],
    "Prob_1": probability[:, 1]
})
```

Bagian ini membuat tabel hasil yang akan dikirim kembali ke KNIME. Kolom `Actual` berisi nilai asli dari dataset, kolom `Prediction` berisi hasil prediksi model, kolom `Prob_0` berisi probabilitas kelas 0, dan kolom `Prob_1` berisi probabilitas kelas 1.

```python
knio.output_tables[0] = knio.Table.from_pandas(output)
```

Kode terakhir ini digunakan untuk mengirim hasil dari Python kembali ke KNIME agar bisa diteruskan ke node **Scorer**.

**Hasil / Output :**

![](https://cdn.mathpix.com/snip/images/K2y7WTgUzOswgegyow_6RvpHWIUpnHwSKGW4-Lt3kJg.original.fullsize.png)

### g. Scorer  
Digunakan untuk mengevaluasi hasil prediksi.

Setting:
- First column = Actual
- Second column = Prediction

![](https://cdn.mathpix.com/snip/images/QC1nY1_xPoD1DO92slYUnULXD49_o6M62AJqaxc8vG8.original.fullsize.png)

![](https://cdn.mathpix.com/snip/images/8w_d_xEazi-NJqFLNtejUEjJaNbsHV0C4O8grTuYagI.original.fullsize.png)

## 8. Hasil Model

Contoh hasil:

| Actual | Prediction | Prob_0 | Prob_1 |
|--------|-----------|--------|--------|
| 1 | 1 | 0.138 | 0.862 |
| 0 | 0 | 0.905 | 0.095 |

Model memilih kelas berdasarkan probabilitas terbesar.

## 9. Confusion Matrix

| Actual / Prediction | 0 | 1 |
|--------------------|---|---|
| 0 | 8108 | 2057 |
| 1 | 1342 | 7806 |


## 10. Perhitungan Evaluasi

Setelah model selesai melakukan prediksi, langkah berikutnya adalah mengevaluasi seberapa baik performa model tersebut. Evaluasi ini penting karena kita ingin mengetahui apakah model benar-benar bisa memprediksi churn dengan baik, bukan sekadar menghasilkan angka saja.

Untuk itu digunakan beberapa metrik evaluasi, yaitu **Accuracy, Precision, dan Recall**. Masing-masing memiliki fungsi yang berbeda dan tidak bisa hanya melihat satu saja.

---

### 10.1 Accuracy

$
Accuracy = \frac{TP + TN}{Total}
$

$
= \frac{7806 + 8108}{19313}
= 0.824
$

≈ 82.4%

**Penjelasan:**

Accuracy digunakan untuk mengukur seberapa banyak prediksi yang benar dibandingkan dengan seluruh data yang diuji. Dengan kata lain, accuracy menunjukkan tingkat ketepatan model secara keseluruhan.

Pada hasil di atas, nilai accuracy sebesar 82.4% berarti dari seluruh data testing, sekitar 82% berhasil diprediksi dengan benar oleh model.

**Kegunaan:**

- Memberikan gambaran umum performa model
- Mudah dipahami karena langsung menunjukkan persentase kebenaran
- Cocok digunakan jika data relatif seimbang antara kelas 0 dan 1

**Catatan:**

Accuracy bisa menipu jika data tidak seimbang. Misalnya jika data lebih banyak yang tidak churn, model bisa saja terlihat bagus hanya karena sering menebak tidak churn.


### 10.2 Precision

$
Precision = \frac{TP}{TP + FP}
$

$
= \frac{7806}{7806 + 2057}
= 0.791
$

**Penjelasan:**

Precision digunakan untuk mengukur seberapa tepat prediksi model ketika mengatakan suatu pelanggan akan churn.

Artinya, dari semua data yang diprediksi sebagai churn, berapa banyak yang benar-benar churn.

Nilai precision sebesar 0.791 berarti sekitar 79.1% dari pelanggan yang diprediksi churn memang benar-benar churn.

**Kegunaan:**

- Penting ketika kesalahan prediksi churn (false positive) harus diminimalkan
- Berguna jika ingin menghindari tindakan yang tidak perlu

**Contoh kasus:**

Jika perusahaan ingin memberikan promo khusus ke pelanggan yang diprediksi churn, maka precision penting.  
Kalau precision rendah, banyak pelanggan yang sebenarnya tidak churn malah diberi promo (boros biaya).


### 10.3 Recall

$
Recall = \frac{TP}{TP + FN}
$

$
= \frac{7806}{7806 + 1342}
= 0.853
$

**Penjelasan:**

Recall digunakan untuk mengukur kemampuan model dalam menemukan semua pelanggan yang benar-benar churn.

Artinya, dari seluruh pelanggan yang memang churn, berapa banyak yang berhasil terdeteksi oleh model.

Nilai recall sebesar 0.853 berarti sekitar 85.3% pelanggan yang churn berhasil dikenali oleh model.

**Kegunaan:**

- Penting ketika ingin menangkap sebanyak mungkin kasus churn
- Berguna jika kehilangan pelanggan lebih berbahaya dibanding salah prediksi

**Contoh kasus:**

Jika perusahaan ingin mencegah kehilangan pelanggan, recall lebih penting.  
Kalau recall rendah, banyak pelanggan yang sebenarnya churn tetapi tidak terdeteksi, sehingga tidak ditangani.

## 10.4 Kesimpulan Evaluasi

Dari ketiga metrik tersebut bisa disimpulkan bahwa:

- Accuracy menunjukkan model sudah cukup baik secara umum
- Precision menunjukkan prediksi churn cukup akurat
- Recall menunjukkan model cukup mampu mendeteksi pelanggan churn

Dalam kasus churn, biasanya **recall lebih penting dibanding precision**, karena lebih baik salah sedikit (false alarm) daripada kehilangan pelanggan tanpa diketahui.

Namun idealnya, model memiliki keseimbangan antara precision dan recall agar tidak terlalu banyak kesalahan di kedua sisi.

## 11. Analisis

Model menunjukkan performa yang cukup baik dengan accuracy di atas 80%. Kemampuan model dalam mendeteksi pelanggan churn juga cukup tinggi, terlihat dari nilai recall yang cukup besar.

Namun masih terdapat kesalahan prediksi, terutama pada false positive dan false negative. Hal ini wajar karena asumsi Naive Bayes yang menganggap fitur tidak saling berhubungan.


## 12. Kesimpulan

Dari hasil analisis yang dilakukan, dapat disimpulkan bahwa metode Naive Bayes dapat digunakan untuk memprediksi churn pelanggan dengan hasil yang cukup baik. Model ini sederhana, cepat, dan tidak membutuhkan proses yang terlalu kompleks.

Meskipun memiliki keterbatasan pada asumsi independensi fitur, hasil yang diperoleh tetap cukup layak digunakan sebagai dasar analisis awal dalam memahami perilaku pelanggan.